# LeetCode 1003 — Check If Word Is Valid After Substitutions bằng Python

Link đề: https://leetcode.com/problems/check-if-word-is-valid-after-substitutions/ · Độ khó: **Medium** · Chủ đề: String (chuỗi), Stack (ngăn xếp)

Skeleton LeetCode:

```python
class Solution:
    def isValid(self, s: str) -> bool:
        ...
```

Ghi chú: xuất phát từ chuỗi rỗng, mỗi lần được **chèn chuỗi `"abc"` vào một vị trí bất kỳ** của chuỗi hiện tại (đầu, giữa hoặc cuối). Chuỗi `s` gọi là **hợp lệ** nếu tạo được từ chuỗi rỗng bằng một số lần chèn như vậy. Làm **ngược lại**: nếu `s` hợp lệ thì có một lần chèn cuối cùng, tức `s` chứa chuỗi con `"abc"`; xóa nó đi ta nhận lại một chuỗi hợp lệ ngắn hơn. Như vậy bài toán trở thành "xóa liên tục chuỗi con `abc` có xóa hết được không?", và **ngăn xếp** làm việc đó trong `O(n)`: đẩy từng ký tự, hễ ba ký tự trên cùng là `abc` thì bỏ ra. Bài này giống bài kiểm tra ngoặc hợp lệ, nhưng với "ngoặc" dài ba ký tự.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tính hợp lệ định nghĩa bằng phép **chèn** `"abc"`, và hiểu vì sao có thể đổi sang phép **xóa** `"abc"`.
2. Nắm các công cụ Python cần thiết: `str.replace`, `in` cho chuỗi con, `list` làm ngăn xếp (`append`, `pop`, lát cắt `stack[-3:]`), `set` các chuỗi sinh ra, và so sánh danh sách với chuỗi.
3. Hiểu nguyên lý: **mọi cách xóa** `abc` (chuỗi con) đều dẫn đến cùng kết quả "xóa hết được hay không" (tính hợp lệ **không phụ thuộc thứ tự xóa**); ngăn xếp thực hiện việc xóa theo một thứ tự cố định trong `O(n)`.
4. Đi qua 4 cách: sinh mọi chuỗi hợp lệ bằng phép chèn rồi tra cứu (brute force, oracle cho chuỗi ngắn), lặp `replace("abc", "")` (nhanh gọn nhưng `O(n^2)`), ngăn xếp (cách nộp chính, `O(n)`), và bộ phân tích đệ quy theo ngữ pháp.
5. Tránh các lỗi kinh điển: chỉ đếm số lượng `a`, `b`, `c` bằng nhau, đếm theo tiền tố mà bỏ qua thứ tự, chỉ `replace` **một lần**, kiểm tra độ dài chia hết cho 3 là đủ, và bỏ qua việc kiểm tra các ký tự **ngay dưới** đỉnh ngăn xếp khi gặp `c`.

## 2. Đề bài & ví dụ

Cho chuỗi `s`, hãy xác định `s` có **hợp lệ** hay không.

Chuỗi `s` **hợp lệ** nếu, bắt đầu từ chuỗi rỗng `t = ""`, ta có thể biến `t` thành `s` sau khi thực hiện phép sau **một số lần bất kỳ**:

- **Chèn** chuỗi `"abc"` vào **vị trí bất kỳ** của `t`. Chính xác hơn, `t` trở thành `t_left + "abc" + t_right` với `t == t_left + t_right` (`t_left` và `t_right` có thể rỗng).

Trả về `true` nếu `s` hợp lệ, ngược lại `false`.

**Ví dụ 1:** `s = "aabcbc"` -> `true`. `"" -> "abc" -> "aabcbc"` (chèn `abc` sau chữ `a` đầu tiên).

**Ví dụ 2:** `s = "abcabcababcc"` -> `true`. `"" -> "abc" -> "abcabc" -> "abcabcabc" -> "abcabcababcc"`.

**Ví dụ 3:** `s = "abccba"` -> `false`. Không thể tạo `"abccba"` bằng phép chèn.

**Ràng buộc:**

- `1 <= s.length <= 2 * 10^4`
- `s` chỉ gồm các chữ `'a'`, `'b'`, `'c'`.

In [1]:
from typing import List, Set, Optional
from collections import Counter
import random, time, sys

examples = [
    ("aabcbc", True),
    ("abcabcababcc", True),
    ("abccba", False),
]
for s, expected in examples:
    print(f"s={s!r} -> kỳ vọng {expected}")

s='aabcbc' -> kỳ vọng True
s='abcabcababcc' -> kỳ vọng True
s='abccba' -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. `str.replace` và chuỗi con

`s.replace("abc", "")` xóa **mọi** lần xuất hiện **không chồng lấn** của `"abc"` trong **một lượt quét**, và trả về chuỗi mới (chuỗi Python bất biến). Sau khi xóa, các ký tự hai bên chỗ vừa xóa **dính lại** với nhau và có thể tạo ra `"abc"` mới, nên một lượt chưa chắc đủ. `"abc" in s` kiểm tra có chứa chuỗi con `abc` không.

In [2]:
s = "aabcbc"
print(repr(s.replace("abc", "")), "<- một lượt: còn lại 'abc' (mới sinh ra do dính lại)")
print(repr(s.replace("abc", "").replace("abc", "")), "<- hai lượt: rỗng")
print("abc" in "acbabc", "abc" in "acb")

'abc' <- một lượt: còn lại 'abc' (mới sinh ra do dính lại)
'' <- hai lượt: rỗng
True False


### 3.2. Danh sách làm ngăn xếp và lát cắt đỉnh

`stack.append(x)` đẩy, `stack.pop()` lấy ra. Ba phần tử trên cùng là `stack[-3:]` (lát cắt không lỗi dù ngăn xếp còn ngắn hơn 3). Ta so sánh `stack[-3:] == ["a", "b", "c"]` hoặc ghép `"".join(stack[-3:]) == "abc"`.

In [3]:
stack = []
for ch in "aab":
    stack.append(ch)
print(stack, "| stack[-3:] =", stack[-3:], "| 2 phần tử:", ["x", "y"][-3:])
stack.append("c")
print(stack[-3:] == ["a", "b", "c"], "-> bỏ ba phần tử:", end=" ")
del stack[-3:]
print(stack)

['a', 'a', 'b'] | stack[-3:] = ['a', 'a', 'b'] | 2 phần tử: ['x', 'y']
True -> bỏ ba phần tử: ['a']


### 3.3. Phép xóa là "ngược" của phép chèn

Nếu `s` được tạo bằng nhiều lần chèn, thì lần chèn **cuối cùng** để lại một chuỗi con `abc` trong `s`; xóa nó cho ra chuỗi trước đó (cũng hợp lệ). Ngược lại, nếu `s` chứa `abc` và sau khi xóa còn chuỗi hợp lệ thì `s` hợp lệ (chèn `abc` trở lại). Từ đó: `s` hợp lệ ⇔ `s` rỗng, hoặc `s` chứa `abc` và có **một** cách xóa dẫn tới chuỗi hợp lệ. Mục 10 chứng minh rằng cách xóa **nào** cũng được.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `s = "aabcbc"`. Chuỗi con `abc` xuất hiện ở vị trí `1..3` (`a a b c b c`, ba ký tự giữa). Xóa: `"a" + "bc"` = `"abc"`. Lại xóa `abc`: rỗng. Vậy hợp lệ.

Ví dụ 3: `s = "abccba"`: xóa `abc` ở đầu còn `"cba"`, không còn `abc` mà chưa rỗng: không hợp lệ. (Và `abccba` không có cách xóa nào khác.)

Theo dõi bằng **ngăn xếp** với `"aabcbc"`:

| Ký tự | Ngăn xếp sau khi đẩy | Ba phần tử trên cùng là `abc`? | Ngăn xếp sau |
|---|---|---|---|
| `a` | `[a]` | không | `[a]` |
| `a` | `[a, a]` | không | `[a, a]` |
| `b` | `[a, a, b]` | không (`aab`) | `[a, a, b]` |
| `c` | `[a, a, b, c]` | có (`abc`) | `[a]` |
| `b` | `[a, b]` | không | `[a, b]` |
| `c` | `[a, b, c]` | có | `[]` |

Ngăn xếp rỗng ở cuối: hợp lệ.

## 5. Cách 1 — Brute Force: sinh mọi chuỗi hợp lệ bằng phép chèn

**Ý tưởng.** Làm đúng theo **định nghĩa**: bắt đầu từ `{""}`; lặp: với mỗi chuỗi `t` đã có và **mỗi vị trí** `i` từ `0` đến `len(t)`, sinh `t[:i] + "abc" + t[i:]` (nếu chưa vượt độ dài `n = len(s)`). Khi không sinh thêm được nữa, ta có tập **mọi chuỗi hợp lệ có độ dài `<= n`**, và `s` hợp lệ ⇔ `s` thuộc tập đó. Không hề dùng phép xóa hay ngăn xếp, nên đây là **oracle độc lập**. Số chuỗi hợp lệ tăng rất nhanh theo độ dài, nên chỉ dùng cho `n` nhỏ.

In [4]:
def generate_valid(max_len: int, word: str = "abc") -> Set[str]:
    """Mọi chuỗi tạo được từ "" bằng phép chèn word vào vị trí bất kỳ, độ dài <= max_len."""
    seen = {""}
    frontier = [""]
    while frontier:
        nxt = []
        for t in frontier:
            if len(t) + len(word) > max_len:
                continue
            for i in range(len(t) + 1):
                u = t[:i] + word + t[i:]
                if u not in seen:
                    seen.add(u)
                    nxt.append(u)
        frontier = nxt
    return seen

valid_12 = generate_valid(12)
print("số chuỗi hợp lệ có độ dài <= 12:", len(valid_12), "| theo độ dài:", sorted(Counter(len(x) for x in valid_12).items()))

def is_valid_brute(s: str) -> bool:
    return s in generate_valid(len(s))

for s, expected in examples:
    got = is_valid_brute(s)
    print(f"s={s!r}: brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

số chuỗi hợp lệ có độ dài <= 12: 72 | theo độ dài: [(0, 1), (3, 1), (6, 3), (9, 12), (12, 55)]
s='aabcbc': brute = True OK
s='abcabcababcc': brute = True OK
s='abccba': brute = False OK


**Độ phức tạp và nút thắt.** Số chuỗi hợp lệ độ dài `3k` tăng rất nhanh (ở trên: 1, 1, 3, 12, 55 chuỗi cho độ dài `0, 3, 6, 9, 12`, và 345 chuỗi có độ dài `<= 15`; mỗi bước độ dài tăng thêm 3 thì số chuỗi nhân lên vài lần); với `n = 2 * 10^4` là không thể. Đo thử:

In [5]:
for n in (6, 9, 12, 15):
    t0 = time.perf_counter()
    cnt = len(generate_valid(n))
    print(f"n <= {n:2}: {cnt:7} chuỗi hợp lệ, {(time.perf_counter() - t0) * 1000:9.1f} ms")
print("n = 20000: không thể")

n <=  6:       5 chuỗi hợp lệ,       0.0 ms
n <=  9:      17 chuỗi hợp lệ,       0.0 ms
n <= 12:      72 chuỗi hợp lệ,       0.0 ms
n <= 15:     345 chuỗi hợp lệ,       0.2 ms
n = 20000: không thể


Nút thắt: ta sinh **mọi** chuỗi hợp lệ để trả lời **một** câu hỏi. Hãy đi ngược: **xóa** `abc` khỏi `s`.

## 6. Ý tưởng thuật toán / bước đột phá

**Xóa liên tục chuỗi con `abc`; hợp lệ ⇔ xóa hết được.**

- Nếu `s` hợp lệ thì lần chèn cuối cùng để lại một `abc` trong `s`; xóa nó được chuỗi hợp lệ ngắn hơn: tiếp tục tới khi rỗng.
- Nếu `s` rỗng sau một chuỗi các lần xóa `abc`, đảo ngược chuỗi thao tác (chèn lại) cho ta cách tạo `s`.

Vậy **hợp lệ ⇔ có một dãy xóa `abc` đưa `s` về rỗng**. Mục 10 chứng minh dãy xóa **không quan trọng** (cứ xóa tùy ý, kết quả không đổi), nên ta có thể xóa theo cách tiện nhất:

| Cách | Ý tưởng |
|---|---|
| 2 | Lặp `s = s.replace("abc", "")` đến khi không đổi |
| 3 | Ngăn xếp: đẩy ký tự, bỏ ba ký tự trên cùng khi thành `abc` (nộp) |
| 4 | Bộ phân tích đệ quy theo ngữ pháp `S -> (a S b S c)*` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Lặp `replace("abc", "")`

**Giải thích từng đoạn.**

- Trong khi `s` còn chứa `"abc"`: thay mọi `"abc"` bằng chuỗi rỗng.
- Cuối cùng `s` rỗng ⇔ hợp lệ.
- Mỗi lượt xóa ít nhất một `abc` nên tối đa `n / 3` lượt; mỗi lượt `O(n)`.

In [6]:
def is_valid_replace(s: str) -> bool:
    while "abc" in s:
        s = s.replace("abc", "")
    return s == ""

for s, expected in examples:
    got = is_valid_replace(s)
    print(f"s={s!r}: replace = {got}", "OK" if got == expected else "SAI")
    assert got == expected

s='aabcbc': replace = True OK
s='abcabcababcc': replace = True OK
s='abccba': replace = False OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def isValid(self, s: str) -> bool:
        while "abc" in s:
            s = s.replace("abc", "")      # xóa mọi "abc" không chồng lấn, lặp đến khi hết
        return s == ""

sol_replace = Solution().isValid
for s, expected in examples:
    assert sol_replace(s) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n^2)` tồi nhất (chuỗi lồng nhau như `a...a bc...bc` cần `n / 3` lượt, mỗi lượt `O(n)`), nhưng hằng số rất nhỏ vì `replace` viết bằng C; bộ nhớ `O(n)`.

---

### 7.2. Cách 3 — Ngăn xếp (cách nộp chính)

**Giải thích từng đoạn.**

- Duyệt từng ký tự `ch`, đẩy vào ngăn xếp.
- Nếu `ch == "c"` và hai phần tử ngay dưới là `"b"` rồi `"a"` (tức ba ký tự trên cùng là `abc`) thì **bỏ cả ba**.
- Cuối cùng hợp lệ ⇔ ngăn xếp rỗng.
- Chỉ cần kiểm tra khi vừa đẩy `c`: `abc` mới hình thành chỉ có thể kết thúc ở ký tự mới đẩy.

In [8]:
def is_valid_stack(s: str) -> bool:
    stack: List[str] = []
    for ch in s:
        stack.append(ch)
        if ch == "c" and stack[-3:] == ["a", "b", "c"]:
            del stack[-3:]
    return not stack

for s, expected in examples:
    got = is_valid_stack(s)
    print(f"s={s!r}: ngăn xếp = {got}", "OK" if got == expected else "SAI")
    assert got == expected

s='aabcbc': ngăn xếp = True OK
s='abcabcababcc': ngăn xếp = True OK
s='abccba': ngăn xếp = False OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def isValid(self, s: str) -> bool:
        stack = []
        for ch in s:
            stack.append(ch)
            if ch == "c" and stack[-3:] == ["a", "b", "c"]:   # ba ký tự trên cùng là "abc"
                del stack[-3:]
        return not stack

sol_stack = Solution().isValid
for s, expected in examples:
    assert sol_stack(s) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** mỗi ký tự được đẩy một lần và bị bỏ nhiều nhất một lần: thời gian `O(n)`, bộ nhớ `O(n)`. Một mẹo: nếu `len(s) % 3 != 0` thì trả về `False` ngay (không bắt buộc nhưng tiết kiệm).

---

### 7.3. Cách 4 — Bộ phân tích đệ quy theo ngữ pháp

**Ý tưởng khác.** Có thể mô tả mọi chuỗi hợp lệ bằng một **ngữ pháp**: `S -> (a S b S c)*`, tức "không hoặc nhiều khối, mỗi khối là `a`, một `S`, `b`, một `S`, `c`". Một bộ phân tích đệ quy xuống (recursive descent) đọc `s` từ trái sang phải: `parse_S()` lặp khi ký tự hiện tại là `a`: đọc `a`, gọi `parse_S()`, đòi `b`, gọi `parse_S()`, đòi `c`. Chuỗi hợp lệ ⇔ phân tích hết đúng toàn bộ chuỗi mà không bị kẹt.

**Giải thích từng đoạn.**

- `pos` là vị trí đang đọc (dùng danh sách một phần tử để sửa trong hàm lồng).
- `parse_S` trả về `True` nếu phân tích khối thành công, `False` nếu gặp ký tự sai chỗ.
- Độ sâu đệ quy tới `n / 3` nên cần nâng giới hạn đệ quy khi chuỗi dài; cách này chủ yếu để thấy **ngữ pháp** của bài và liên hệ với ngăn xếp.

In [10]:
def is_valid_parser(s: str) -> bool:
    n = len(s)
    pos = [0]
    sys.setrecursionlimit(max(sys.getrecursionlimit(), 3 * n + 1000))

    def parse_S() -> bool:
        while pos[0] < n and s[pos[0]] == "a":
            pos[0] += 1                              # a
            if not parse_S():                        # S
                return False
            if pos[0] >= n or s[pos[0]] != "b":      # b
                return False
            pos[0] += 1
            if not parse_S():                        # S
                return False
            if pos[0] >= n or s[pos[0]] != "c":      # c
                return False
            pos[0] += 1
        return True

    return parse_S() and pos[0] == n

for s, expected in examples:
    got = is_valid_parser(s)
    print(f"s={s!r}: bộ phân tích = {got}", "OK" if got == expected else "SAI")
    assert got == expected

s='aabcbc': bộ phân tích = True OK
s='abcabcababcc': bộ phân tích = True OK
s='abccba': bộ phân tích = False OK


**Code nộp LeetCode (Cách 4):** (đúng nhưng đệ quy sâu: chuỗi lồng nhau `n / 3` mức; chỉ để minh họa)

In [11]:
class Solution:
    def isValid(self, s: str) -> bool:
        import sys
        n = len(s)
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 3 * n + 1000))
        pos = [0]

        def parse_S() -> bool:                       # S -> (a S b S c)*
            while pos[0] < n and s[pos[0]] == "a":
                pos[0] += 1
                if not parse_S():
                    return False
                if pos[0] >= n or s[pos[0]] != "b":
                    return False
                pos[0] += 1
                if not parse_S():
                    return False
                if pos[0] >= n or s[pos[0]] != "c":
                    return False
                pos[0] += 1
            return True

        return parse_S() and pos[0] == n

sol_parser = Solution().isValid
for s, expected in examples:
    assert sol_parser(s) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n)`, bộ nhớ `O(n)` cho ngăn xếp đệ quy (độ sâu tới `n / 3`; với `n = 2 * 10^4` là hàng nghìn mức: sát giới hạn, nên cách này dễ gặp lỗi đệ quy ở môi trường chấm; ngăn xếp tường minh an toàn hơn).

## 8. TRACE từng bước

### 8.1. Ngăn xếp cho `"abcabcababcc"` (ví dụ 2)

In ký tự, ngăn xếp sau khi đẩy và sau khi bỏ.

In [12]:
def trace_stack(s):
    stack = []
    for i, ch in enumerate(s):
        stack.append(ch)
        before = "".join(stack)
        popped = ch == "c" and stack[-3:] == ["a", "b", "c"]
        if popped:
            del stack[-3:]
        print(f"i={i:2} ch={ch!r}: đẩy -> {before!r:14}" + (" -> có abc, bỏ ->" if popped else " -> không bỏ ->") + f" {''.join(stack)!r}")
    return not stack

assert trace_stack("abcabcababcc") is True

i= 0 ch='a': đẩy -> 'a'            -> không bỏ -> 'a'
i= 1 ch='b': đẩy -> 'ab'           -> không bỏ -> 'ab'
i= 2 ch='c': đẩy -> 'abc'          -> có abc, bỏ -> ''
i= 3 ch='a': đẩy -> 'a'            -> không bỏ -> 'a'
i= 4 ch='b': đẩy -> 'ab'           -> không bỏ -> 'ab'
i= 5 ch='c': đẩy -> 'abc'          -> có abc, bỏ -> ''
i= 6 ch='a': đẩy -> 'a'            -> không bỏ -> 'a'
i= 7 ch='b': đẩy -> 'ab'           -> không bỏ -> 'ab'
i= 8 ch='a': đẩy -> 'aba'          -> không bỏ -> 'aba'
i= 9 ch='b': đẩy -> 'abab'         -> không bỏ -> 'abab'
i=10 ch='c': đẩy -> 'ababc'        -> có abc, bỏ -> 'ab'
i=11 ch='c': đẩy -> 'abc'          -> có abc, bỏ -> ''


### 8.2. Một ví dụ không hợp lệ: `"abccba"`

Ngăn xếp cuối cùng còn `"cba"` (không rỗng).

In [13]:
assert trace_stack("abccba") is False

i= 0 ch='a': đẩy -> 'a'            -> không bỏ -> 'a'
i= 1 ch='b': đẩy -> 'ab'           -> không bỏ -> 'ab'
i= 2 ch='c': đẩy -> 'abc'          -> có abc, bỏ -> ''
i= 3 ch='c': đẩy -> 'c'            -> không bỏ -> 'c'
i= 4 ch='b': đẩy -> 'cb'           -> không bỏ -> 'cb'
i= 5 ch='a': đẩy -> 'cba'          -> không bỏ -> 'cba'


### 8.3. Lặp `replace` cho chuỗi lồng nhau `"aaabcbcbc"`

Mỗi lượt chỉ xóa lớp trong cùng.

In [14]:
s = "aaabcbcbc"
step = 0
while "abc" in s:
    new = s.replace("abc", "")
    print(f"lượt {step}: {s!r} -> {new!r}")
    s, step = new, step + 1
print("cuối:", repr(s), "-> hợp lệ:", s == "")
assert s == ""

lượt 0: 'aaabcbcbc' -> 'aabcbc'
lượt 1: 'aabcbc' -> 'abc'
lượt 2: 'abc' -> ''
cuối: '' -> hợp lệ: True


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (ngăn xếp): `O(n)` thời gian, ngắn gọn.

```python
class Solution:
    def isValid(self, s: str) -> bool:
        stack = []
        for ch in s:
            stack.append(ch)
            if ch == "c" and stack[-3:] == ["a", "b", "c"]:   # ba ký tự trên cùng là "abc"
                del stack[-3:]
        return not stack
```

In [15]:
class Solution:
    def isValid(self, s: str) -> bool:
        stack = []
        for ch in s:
            stack.append(ch)
            if ch == "c" and stack[-3:] == ["a", "b", "c"]:   # ba ký tự trên cùng là "abc"
                del stack[-3:]
        return not stack

sol = Solution()
for s, expected in examples:
    got = sol.isValid(s)
    print(f"s={s!r}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

s='aabcbc': True OK
s='abcabcababcc': True OK
s='abccba': False OK


## 10. Vì sao đúng

**Hai chiều của việc đổi chèn sang xóa.**

- *Hợp lệ ⇒ xóa hết được.* Nếu `s` tạo từ `""` bằng `k` lần chèn, lần chèn cuối để lại `abc` trong `s`; xóa nó được chuỗi tạo bằng `k - 1` lần chèn; quy nạp theo `k`.
- *Xóa hết được ⇒ hợp lệ.* Nếu một dãy `k` lần xóa `abc` đưa `s` về rỗng, thì chèn lại theo thứ tự ngược cho `s` từ `""`.

**Dãy xóa không quan trọng (hội tụ).** Hệ luật viết lại `abc -> ""` **kết thúc** (mỗi lần xóa làm chuỗi ngắn đi) và **hội tụ cục bộ**: hai chuỗi con `abc` khác nhau trong cùng một chuỗi **không bao giờ chồng lấn** (ba chữ `a`, `b`, `c` khác nhau nên hai lần xuất hiện chồng lên nhau sẽ buộc một ký tự vừa là `a` vừa là `b` hoặc `c`), vì vậy xóa chúng theo thứ tự nào cũng cho cùng một kết quả. Theo bổ đề Newman (kết thúc + hội tụ cục bộ ⇒ hội tụ), mỗi chuỗi có **đúng một** dạng rút gọn cuối cùng, bất kể thứ tự xóa. Do đó `s` hợp lệ ⇔ dạng rút gọn của `s` là chuỗi rỗng ⇔ **mọi** dãy xóa tham lam (kể cả của ngăn xếp hay `replace`) đều đưa `s` về rỗng. Ta còn kiểm chứng tính chất này bằng máy: với mọi chuỗi hợp lệ độ dài `<= 12`, xóa `abc` ở **bất kỳ** vị trí nào vẫn cho chuỗi hợp lệ (cell dưới).

**Ngăn xếp thực hiện một thứ tự xóa hợp lệ.** Ngăn xếp luôn bằng chuỗi `s[:i]` đã được xóa tối đa bằng các lần xóa `abc` kết thúc ở các ký tự đã đọc. Khi đẩy `c` và ba ký tự trên cùng là `abc` thì đó là một chuỗi con `abc` thật sự của chuỗi hiện tại, ta xóa nó. Khi duyệt hết, không còn `abc` nào mà ta chưa xóa **đã kết thúc ở ký tự đã đọc** nên chuỗi còn lại (ngăn xếp ghép với phần chưa đọc là rỗng) không còn `abc`; vì dãy xóa không quan trọng, `s` hợp lệ ⇔ ngăn xếp rỗng.

**Độ phức tạp.** Mỗi ký tự đẩy một lần, bỏ nhiều nhất một lần: `O(n)`.

**Kiểm chứng bằng máy.** (1) tập hợp mọi chuỗi hợp lệ độ dài `<= 12` sinh bằng phép **chèn** (Cách 1); (2) **mọi** chuỗi trên `{a,b,c}` độ dài `<= 10` (88 573 chuỗi) được đối chiếu với tập này; (3) tính chất hội tụ: xóa `abc` ở **bất kỳ** vị trí nào từ chuỗi hợp lệ cho chuỗi hợp lệ.

In [16]:
valid_set = generate_valid(12)

# (3) hội tụ: từ chuỗi hợp lệ, xóa một abc ở BẤT KỲ vị trí nào cho chuỗi hợp lệ
for s in valid_set:
    start = s.find("abc")
    while start != -1:
        assert s[:start] + s[start + 3:] in valid_set, (s, start)
        start = s.find("abc", start + 1)
print("(3) hội tụ: với mọi chuỗi hợp lệ độ dài <= 12, xóa abc ở bất kỳ vị trí nào vẫn hợp lệ:", len(valid_set), "chuỗi")

# (2) mọi chuỗi độ dài <= 10 trên {a, b, c}
from itertools import product
count = 0
for L in range(0, 11):
    for tup in product("abc", repeat=L):
        s = "".join(tup)
        expected = s in valid_set
        assert sol_stack(s) == expected, s
        assert sol_replace(s) == expected, s
        count += 1
print("(2) đã đối chiếu", count, "chuỗi (mọi chuỗi độ dài <= 10): ngăn xếp và replace khớp định nghĩa")

(3) hội tụ: với mọi chuỗi hợp lệ độ dài <= 12, xóa abc ở bất kỳ vị trí nào vẫn hợp lệ: 72 chuỗi


(2) đã đối chiếu 88573 chuỗi (mọi chuỗi độ dài <= 10): ngăn xếp và replace khớp định nghĩa


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Sinh mọi chuỗi hợp lệ | rất lớn (hàm mũ) | rất lớn |
| 2. Lặp `replace` | `O(n^2)` tồi nhất | `O(n)` |
| 3. Ngăn xếp | `O(n)` | `O(n)` |
| 4. Bộ phân tích đệ quy | `O(n)` | `O(n)` (độ sâu đệ quy) |

Đo ở cỡ tối đa (`n = 2 * 10^4`) với ba dạng: lồng nhau sâu nhất, xếp liền (`abcabc...`), và ngẫu nhiên hợp lệ:

In [17]:
def nested(k):                      # a^k (bc)^k: lồng nhau sâu nhất, hợp lệ, độ dài 3k
    return "a" * k + "bc" * k

def flat(k):                        # (abc)^k
    return "abc" * k

def planted(rng, k):                # chèn "abc" tại vị trí ngẫu nhiên k lần: chuỗi hợp lệ độ dài 3k
    t = []
    for _ in range(k):
        i = rng.randint(0, len(t))
        t[i:i] = ["a", "b", "c"]
    return "".join(t)

rng = random.Random(11)
k = 6666                           # độ dài 19998 <= 2 * 10^4
cases = {"lồng sâu nhất": nested(k), "xếp liền": flat(k), "ngẫu nhiên hợp lệ": planted(rng, k)}
for label, s in cases.items():
    assert len(s) <= 2 * 10 ** 4
    for name, fn in (("Cách 2", sol_replace), ("Cách 3", sol_stack), ("Cách 4", sol_parser)):
        t0 = time.perf_counter()
        got = fn(s)
        dt = (time.perf_counter() - t0) * 1000
        assert got is True, (label, name)
        print(f"{label:20} {name}: {dt:8.2f} ms (kết quả {got})")

lồng sâu nhất        Cách 2:   240.17 ms (kết quả True)
lồng sâu nhất        Cách 3:     1.25 ms (kết quả True)
lồng sâu nhất        Cách 4:     2.99 ms (kết quả True)
xếp liền             Cách 2:     0.03 ms (kết quả True)
xếp liền             Cách 3:     1.62 ms (kết quả True)
xếp liền             Cách 4:     2.52 ms (kết quả True)
ngẫu nhiên hợp lệ    Cách 2:     0.33 ms (kết quả True)
ngẫu nhiên hợp lệ    Cách 3:     1.47 ms (kết quả True)
ngẫu nhiên hợp lệ    Cách 4:     2.43 ms (kết quả True)


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle sinh bằng phép chèn), đối chiếu cả bốn cách với oracle, test ngẫu nhiên có seed (đúng ràng buộc, cả chuỗi dài) và tính chất.

In [18]:
def all_solutions():
    return {"replace": is_valid_replace, "stack": is_valid_stack, "parser": is_valid_parser,
            "sol_replace": sol_replace, "sol_stack": sol_stack, "sol_parser": sol_parser, "final": sol.isValid}

def check_all(s, expected=None):
    results = {name: fn(s) for name, fn in all_solutions().items()}
    assert len(set(results.values())) == 1, (s, results)
    ref = results["stack"]
    if len(s) <= 12:
        assert ref == (s in valid_set), (s, ref)
    if expected is not None:
        assert ref is expected, (s, ref, expected)
    return ref

# 12.1 ví dụ đề
for s, expected in examples:
    check_all(s, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ("abc", True),                  # một lần chèn
    ("a", False), ("b", False), ("c", False),   # một ký tự (độ dài không chia hết cho 3)
    ("ab", False), ("bc", False), ("ac", False),
    ("cba", False),                 # đúng số lượng nhưng sai thứ tự
    ("bca", False), ("acb", False), ("bac", False), ("cab", False),
    ("abcabc", True),               # hai khối liền nhau
    ("aabcbc", True),               # ví dụ 1: lồng nhau
    ("aabbcc", False),              # đủ số lượng, tiền tố hợp lệ nhưng không có chuỗi con abc để xóa
    ("ababcc", True),               # "ab"+"abc"+"c": xóa abc ở giữa còn "abc"
    ("abcabcababcc", True),         # ví dụ 2
    ("abccba", False),              # ví dụ 3
    ("ccc", False), ("aaa", False), ("bbb", False),
    ("aabcbcabc", True),            # lồng nhau rồi liền nhau
    ("abcc", False),                # độ dài không chia hết cho 3
    ("aaabbbccc", False),           # đủ số lượng, không có abc
    ("aaabcbcbc", True),            # lồng 3 lớp
]
for s, expected in edge:
    check_all(s, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 25 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm, so với oracle sinh bằng phép chèn)
def wrong_count_equal(s):                        # chỉ đếm a == b == c
    return s.count("a") == s.count("b") == s.count("c")

def wrong_prefix_counts(s):                      # đếm theo tiền tố a >= b >= c và tổng bằng nhau
    a = b = c = 0
    for ch in s:
        if ch == "a":
            a += 1
        elif ch == "b":
            b += 1
        else:
            c += 1
        if not (a >= b >= c):
            return False
    return a == b == c

def wrong_replace_once(s):                       # chỉ replace một lượt
    return s.replace("abc", "") == ""

def wrong_len_mod3(s):                           # chỉ kiểm tra độ dài chia hết cho 3
    return len(s) % 3 == 0

def wrong_stack_no_check_below(s):               # gặp c thì bỏ hai phần tử dưới mà không kiểm tra là b và a
    stack = []
    for ch in s:
        if ch == "c" and len(stack) >= 2:
            stack.pop()
            stack.pop()
        else:
            stack.append(ch)
    return not stack

def wrong_stack_top_only(s):                     # gặp c chỉ kiểm tra đỉnh là b
    stack = []
    for ch in s:
        if ch == "c" and stack and stack[-1] == "b":
            stack.pop()
            if stack:
                stack.pop()
        else:
            stack.append(ch)
    return not stack

def find_counter(wrong, max_len=9):
    for L in range(1, max_len + 1):
        for tup in product("abc", repeat=L):
            s = "".join(tup)
            if wrong(s) != (s in valid_set):
                return s
    return None

counters = {}
for name, fn in (("chỉ đếm a == b == c", wrong_count_equal), ("đếm theo tiền tố", wrong_prefix_counts),
                 ("chỉ replace một lượt", wrong_replace_once), ("chỉ kiểm tra len % 3", wrong_len_mod3),
                 ("bỏ hai phần tử không kiểm tra", wrong_stack_no_check_below), ("chỉ kiểm tra đỉnh là b", wrong_stack_top_only)):
    s = find_counter(fn)
    assert s is not None, name
    counters[name] = s
    print(f"{name:32} s={s!r}: sai -> {fn(s)}, đúng -> {s in valid_set}")
    check_all(s)
print("12.3 phản ví dụ: OK")

chỉ đếm a == b == c              s='acb': sai -> True, đúng -> False
đếm theo tiền tố                 s='aabbcc': sai -> True, đúng -> False
chỉ replace một lượt             s='aabcbc': sai -> False, đúng -> True
chỉ kiểm tra len % 3             s='aaa': sai -> True, đúng -> False
bỏ hai phần tử không kiểm tra    s='aac': sai -> True, đúng -> False
chỉ kiểm tra đỉnh là b           s='bc': sai -> True, đúng -> False
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên: chuỗi hợp lệ cài sẵn (chèn abc), và biến đổi nhỏ (đổi một ký tự / hoán vị hai ký tự) so với oracle bằng cách xóa ngây thơ
def naive_oracle(s):                              # độc lập: thử MỌI vị trí xóa abc (đệ quy có nhớ), không dùng ngăn xếp
    from functools import lru_cache
    @lru_cache(None)
    def f(t):
        if t == "":
            return True
        i = t.find("abc")
        while i != -1:
            if f(t[:i] + t[i + 3:]):
                return True
            i = t.find("abc", i + 1)
        return False
    return f(s)

rng = random.Random(4003)
n_true = n_false = 0
for _ in range(500):
    k = rng.randint(1, 8)
    s = planted(rng, k)
    assert check_all(s, True) is True
    n_true += 1
    # biến đổi nhỏ
    t = list(s)
    if rng.random() < 0.5:
        i = rng.randrange(len(t)); t[i] = rng.choice("abc")
    else:
        i, j = rng.randrange(len(t)), rng.randrange(len(t)); t[i], t[j] = t[j], t[i]
    t = "".join(t)
    ref = naive_oracle(t)
    assert check_all(t, ref) is ref
    n_false += not ref
print(f"12.4 ngẫu nhiên: {n_true} chuỗi hợp lệ cài sẵn, {n_false} chuỗi biến đổi không hợp lệ: OK")

# 12.5 chuỗi dài đúng giới hạn: hợp lệ cài sẵn, rồi phá hỏng nhẹ
rng = random.Random(5003)
for _ in range(5):
    s = planted(rng, 6666)
    assert len(s) == 19998
    assert sol.isValid(s) and sol_replace(s) and sol_parser(s)
    t = list(s)
    i = rng.randrange(len(t) - 1)
    t[i], t[i + 1] = t[i + 1], t[i]               # hoán vị hai ký tự kề nhau
    t = "".join(t)
    ref = sol_replace(t)
    assert sol.isValid(t) == ref == sol_parser(t)
    assert sol.isValid(s + "a") is False and sol.isValid("c" + s) is False
print("12.5 chuỗi dài 19998: OK")

# 12.6 tính chất: ghép hai chuỗi hợp lệ là hợp lệ; bao bởi a ... bc là hợp lệ; thêm một ký tự vào chuỗi hợp lệ thì không hợp lệ (độ dài không chia hết cho 3)
rng = random.Random(6003)
for _ in range(300):
    s1, s2 = planted(rng, rng.randint(0, 10)), planted(rng, rng.randint(0, 10))
    assert sol.isValid(s1 + s2) if (s1 + s2) else True
    if s1:
        assert sol.isValid(s1)
        assert sol.isValid("a" + s1 + "bc") and sol.isValid("ab" + s1 + "c")
        assert not sol.isValid(s1 + rng.choice("abc"))
print("12.6 tính chất: OK")

12.4 ngẫu nhiên: 500 chuỗi hợp lệ cài sẵn, 334 chuỗi biến đổi không hợp lệ: OK
12.5 chuỗi dài 19998: OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm số lượng `a`, `b`, `c` bằng nhau

Số lượng bằng nhau là điều kiện **cần**, không **đủ**: thứ tự mới quyết định.

In [21]:
s = counters["chỉ đếm a == b == c"]
print(f"s={s!r}: sai -> {wrong_count_equal(s)} | đúng -> {Solution().isValid(s)}")
assert wrong_count_equal(s) != Solution().isValid(s)
print("ví dụ 3 'abccba':", wrong_count_equal("abccba"), "(sai) so với", Solution().isValid("abccba"))

s='acb': sai -> True | đúng -> False
ví dụ 3 'abccba': True (sai) so với False


Vì sao sai: `"abccba"` có 2 chữ mỗi loại nhưng không thể tạo bằng phép chèn `abc`. **Đúng:** phải tôn trọng **thứ tự** `a` trước `b` trước `c` (ngăn xếp).

### Lỗi 2 — Đếm theo tiền tố (`a >= b >= c`) rồi tổng bằng nhau

Điều kiện giống "ngoặc hợp lệ" này mạnh hơn lỗi 1 nhưng **vẫn chưa đủ** vì không kiểm tra các khối `abc` liền nhau như một chỉnh thể.

In [22]:
s = counters["đếm theo tiền tố"]
print(f"s={s!r}: sai -> {wrong_prefix_counts(s)} | đúng -> {Solution().isValid(s)}")
assert wrong_prefix_counts(s) != Solution().isValid(s)

s='aabbcc': sai -> True | đúng -> False


Vì sao sai: `"aabbcc"` có mọi tiền tố thỏa `a >= b >= c` và tổng bằng nhau, nhưng không chứa `abc` để xóa. **Đúng:** kiểm tra ba ký tự **liền nhau** `abc` (ngăn xếp).

### Lỗi 3 — Chỉ `replace` một lượt

Sau khi xóa `abc`, các ký tự hai bên dính lại và có thể tạo `abc` mới; cần **lặp** hoặc dùng ngăn xếp.

In [23]:
s = counters["chỉ replace một lượt"]
print(f"s={s!r}: sai -> {wrong_replace_once(s)} | đúng -> {Solution().isValid(s)}")
assert wrong_replace_once(s) != Solution().isValid(s)
print("ví dụ 1 'aabcbc':", wrong_replace_once("aabcbc"), "(sai) so với", Solution().isValid("aabcbc"))

s='aabcbc': sai -> False | đúng -> True
ví dụ 1 'aabcbc': False (sai) so với True


Vì sao sai: `"aabcbc".replace("abc", "")` cho `"abc"`, chưa rỗng. **Đúng:** `while "abc" in s: s = s.replace("abc", "")`.

### Lỗi 4 — Chỉ kiểm tra độ dài chia hết cho 3

Độ dài `3k` là điều kiện cần nhưng rất yếu.

In [24]:
s = counters["chỉ kiểm tra len % 3"]
print(f"s={s!r}: sai -> {wrong_len_mod3(s)} | đúng -> {Solution().isValid(s)}")
assert wrong_len_mod3(s) != Solution().isValid(s)

s='aaa': sai -> True | đúng -> False


Vì sao sai: `"aaa"` có độ dài 3 nhưng không hợp lệ. **Đúng:** dùng độ dài chia hết cho 3 chỉ để loại nhanh các chuỗi sai.

### Lỗi 5 — Gặp `c` mà không kiểm tra các ký tự ngay dưới

Bỏ hai phần tử dưới đỉnh mà không kiểm tra chúng là `b` rồi `a` sẽ xóa nhầm.

In [25]:
s = counters["bỏ hai phần tử không kiểm tra"]
print(f"s={s!r}: sai -> {wrong_stack_no_check_below(s)} | đúng -> {Solution().isValid(s)}")
assert wrong_stack_no_check_below(s) != Solution().isValid(s)
s2 = counters["chỉ kiểm tra đỉnh là b"]
print(f"(chỉ kiểm tra đỉnh là b) s={s2!r}: sai -> {wrong_stack_top_only(s2)} | đúng -> {Solution().isValid(s2)}")
assert wrong_stack_top_only(s2) != Solution().isValid(s2)

s='aac': sai -> True | đúng -> False
(chỉ kiểm tra đỉnh là b) s='bc': sai -> True | đúng -> False


Vì sao sai: khối cần xóa là **ba ký tự liền nhau đúng thứ tự** `a, b, c`. **Đúng:** `stack[-3:] == ["a", "b", "c"]`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chuỗi tạo bằng cách chèn một khối cố định vào chỗ bất kỳ" ⇒ **xóa khối** liên tục; dùng **ngăn xếp** (như kiểm tra ngoặc hợp lệ, bài 20).
- Với khối dài `m` có các ký tự **đôi một khác nhau**, ngăn xếp kiểm tra `m` phần tử trên cùng khi vừa đẩy ký tự cuối của khối; kết quả không phụ thuộc thứ tự xóa.
- Điều kiện "đếm bằng nhau" hay "tiền tố đủ" **không** thay được kiểm tra khối liền nhau.
- `str.replace` lặp gọn nhưng có thể `O(n^2)`: với `n <= 2 * 10^4` vẫn chạy nhanh vì hiện thực bằng C.
- Cùng họ: Valid Parentheses (20), Remove All Adjacent Duplicates In String (1047), Remove Outermost Parentheses (1021), Minimum Remove to Make Valid Parentheses (1249).
- Test nên có: `"abc"`, `"aabcbc"`, `"abccba"`, `"aabbcc"`, `"aaabcbcbc"`, độ dài không chia hết cho 3, chuỗi dài lồng sâu.

## 15. Biến thể

1. **Khối `word` bất kỳ gồm các chữ đôi một khác nhau** (ví dụ `"ab"`, `"abcd"`): ngăn xếp kiểm tra hậu tố bằng `word` (bài tập 5).
2. **Khối là một trong nhiều từ** (ví dụ `"ab"` hoặc `"abc"`): phức tạp hơn vì có thể có nhiều cách xóa dẫn tới kết quả khác nhau.
3. **Đếm số lần chèn** (`len(s) // 3`) hoặc **in dãy thao tác chèn** theo thứ tự ngược của dãy xóa.
4. Bài liên quan: 20, 1047, 1249.

Biến thể 1 kiểm bằng oracle sinh bằng phép chèn của chính `word`:

In [26]:
def is_valid_word(s: str, word: str) -> bool:
    m = len(word)
    stack: List[str] = []
    for ch in s:
        stack.append(ch)
        if ch == word[-1] and len(stack) >= m and "".join(stack[-m:]) == word:
            del stack[-m:]
    return not stack

for word in ("ab", "abc", "abcd", "xy"):
    alphabet = sorted(set(word)) + ["z"]
    valid = generate_valid(10, word)
    for L in range(0, 9):
        for tup in product(alphabet, repeat=L):
            s = "".join(tup)
            assert is_valid_word(s, word) == (s in valid), (word, s)
print("biến thể khối bất kỳ gồm chữ đôi một khác nhau: khớp oracle chèn với các khối 'ab', 'abc', 'abcd', 'xy'")
assert is_valid_word("aabcbc", "abc") == Solution().isValid("aabcbc")

biến thể khối bất kỳ gồm chữ đôi một khác nhau: khớp oracle chèn với các khối 'ab', 'abc', 'abcd', 'xy'


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sinh mọi chuỗi hợp lệ | Theo định nghĩa (chèn) | hàm mũ | hàm mũ | Oracle, `n <= 12` |
| 2. Lặp `replace` | Xóa `abc` đến khi hết | `O(n^2)` | `O(n)` | Viết nhanh, `n` vừa phải |
| 3. Ngăn xếp | Xóa khi vừa hình thành `abc` | `O(n)` | `O(n)` | **Cách nộp chính** |
| 4. Bộ phân tích đệ quy | Ngữ pháp `S -> (a S b S c)*` | `O(n)` | `O(n)` (đệ quy) | Minh họa ngữ pháp |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(repr("aabcbc".replace("abc", "")), repr("abcabc".replace("abc", "")), "abc" in "acbabc")
print(["a", "b"][-3:], ["a", "b", "c", "d"][-3:])
print(Solution().isValid("abcabc"), Solution().isValid("abccba"))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # ba giá trị
my_guess_2 = None      # hai danh sách
my_guess_3 = None      # hai bool

### Bài 2 — Tính tay (dễ-vừa)

Chạy tay Cách 3 (ngăn xếp) trên `s = "aaabcbcbc"`. Ghi **kích thước lớn nhất** mà ngăn xếp đạt được (tính ngay **sau khi đẩy** ký tự, **trước khi** bỏ `abc`) và kết quả cuối cùng (`True`/`False`) dưới dạng bộ `(kích_thước_lớn_nhất, kết_quả)`.

In [28]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành ngăn xếp.

In [29]:
def exercise_3(s: str) -> bool:
    stack = []
    for ch in s:
        # TODO 1: đẩy ch vào stack
        # TODO 2: nếu ch == "c" và ba phần tử trên cùng là a, b, c thì bỏ cả ba
        pass
    # TODO 3: hợp lệ khi stack rỗng
    pass


# for s, expected in examples:
#     assert exercise_3(s) is expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, tự viết `exercise_4(s)` bằng **lặp `replace`**.

In [30]:
def exercise_4(s: str) -> bool:
    # TODO
    pass


# for s, expected in examples:
#     assert exercise_4(s) is expected

### Bài 5 — Mini challenge: khối bất kỳ (khó)

Viết `exercise_5(s, word)` trả về `True` nếu `s` có thể tạo từ chuỗi rỗng bằng cách chèn `word` vào vị trí bất kỳ nhiều lần (`word` gồm các chữ **đôi một khác nhau**).

```python
exercise_5("aabb", "ab")       -> True     # "" -> "ab" -> "aabb"
exercise_5("abab", "ab")       -> True
exercise_5("ba", "ab")         -> False
exercise_5("aabcbc", "abc")    -> True
exercise_5("abcdabcd", "abcd") -> True
exercise_5("abdc", "abcd")     -> False
exercise_5("", "abc")          -> True
```
Gợi ý: khi vừa đẩy ký tự cuối của `word`, kiểm tra `len(word)` phần tử trên cùng.

In [31]:
def exercise_5(s: str, word: str) -> bool:
    # TODO
    pass


# assert exercise_5("aabb", "ab") is True
# assert exercise_5("abab", "ab") is True
# assert exercise_5("ba", "ab") is False
# assert exercise_5("aabcbc", "abc") is True
# assert exercise_5("abcdabcd", "abcd") is True
# assert exercise_5("abdc", "abcd") is False
# assert exercise_5("", "abc") is True

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `replace` xóa các `abc` không chồng lấn trong một lượt; lát cắt `[-3:]` không báo lỗi khi danh sách ngắn.
- Mức 2: `"aabcbc"` xóa `abc` giữa còn `"abc"`; `"abcabc"` xóa cả hai.
- Mức 3: dòng 1: `'abc' '' True`; dòng 2: `['a', 'b'] ['b', 'c', 'd']`; dòng 3: `True False`.

**Bài 2**
- Mức 1: theo dõi từng ký tự: `a a a b c` -> ngăn xếp `[a,a,a,b,c]` rồi bỏ ba.
- Mức 2: sau đó `[a,a]`, đẩy `b` (3), `c` (4) rồi bỏ ba còn `[a]`, ...
- Mức 3: kích thước lớn nhất là `5`; ngăn xếp rỗng cuối cùng: `(5, True)`.

**Bài 3**
- Mức 1: `stack.append(ch)`.
- Mức 2: `if ch == "c" and stack[-3:] == ["a", "b", "c"]: del stack[-3:]`.
- Mức 3: `return not stack`.

**Bài 4**
- Mức 1: `while "abc" in s:`.
- Mức 2: `s = s.replace("abc", "")`.
- Mức 3: `return s == ""`.

**Bài 5**
- Mức 1: `m = len(word)`; đẩy ký tự vào `stack`.
- Mức 2: nếu `ch == word[-1]` và `len(stack) >= m` thì so `"".join(stack[-m:])` với `word`.
- Mức 3: khớp thì `del stack[-m:]`; cuối cùng `return not stack`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
print(repr("aabcbc".replace("abc", "")), repr("abcabc".replace("abc", "")), "abc" in "acbabc")
print(["a", "b"][-3:], ["a", "b", "c", "d"][-3:])
print(Solution().isValid("abcabc"), Solution().isValid("abccba"))

assert ("aabcbc".replace("abc", ""), "abcabc".replace("abc", ""), "abc" in "acbabc") == ("abc", "", True)
assert (["a", "b"][-3:], ["a", "b", "c", "d"][-3:]) == (["a", "b"], ["b", "c", "d"])
assert (Solution().isValid("abcabc"), Solution().isValid("abccba")) == (True, False)

'abc' '' True
['a', 'b'] ['b', 'c', 'd']
True False


### Đáp án Bài 2

Theo dõi ngăn xếp sau khi đẩy (độ dài trước khi bỏ): `a`(1), `a`(2), `a`(3), `b`(4), `c`(5, bỏ ba còn 2), `b`(3), `c`(4, bỏ ba còn 1), `b`(2), `c`(3, bỏ ba còn 0). Kích thước lớn nhất `5`, kết quả cuối `True`.

In [33]:
def max_stack_size(s):
    stack, best = [], 0
    for ch in s:
        stack.append(ch)
        best = max(best, len(stack))
        if ch == "c" and stack[-3:] == ["a", "b", "c"]:
            del stack[-3:]
    return best, not stack

hand_answer_solution = (5, True)
assert max_stack_size("aaabcbcbc") == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_solution(s: str) -> bool:
    stack = []
    for ch in s:
        stack.append(ch)
        if ch == "c" and stack[-3:] == ["a", "b", "c"]:
            del stack[-3:]
    return not stack

for s, expected in examples:
    assert exercise_3_solution(s) is expected
for L in range(0, 9):
    for tup in product("abc", repeat=L):
        s = "".join(tup)
        assert exercise_3_solution(s) == (s in valid_set)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_solution(s: str) -> bool:
    while "abc" in s:
        s = s.replace("abc", "")
    return s == ""

for s, expected in examples:
    assert exercise_4_solution(s) is expected
for L in range(0, 9):
    for tup in product("abc", repeat=L):
        s = "".join(tup)
        assert exercise_4_solution(s) == (s in valid_set)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_solution(s: str, word: str) -> bool:
    m = len(word)
    stack = []
    for ch in s:
        stack.append(ch)
        if ch == word[-1] and len(stack) >= m and "".join(stack[-m:]) == word:
            del stack[-m:]
    return not stack

assert exercise_5_solution("aabb", "ab") is True
assert exercise_5_solution("abab", "ab") is True
assert exercise_5_solution("ba", "ab") is False
assert exercise_5_solution("aabcbc", "abc") is True
assert exercise_5_solution("abcdabcd", "abcd") is True
assert exercise_5_solution("abdc", "abcd") is False
assert exercise_5_solution("", "abc") is True
for word in ("ab", "abc", "abcd"):
    valid = generate_valid(9, word)
    for L in range(0, 8):
        for tup in product(sorted(set(word)) + ["z"], repeat=L):
            s = "".join(tup)
            assert exercise_5_solution(s, word) == (s in valid)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (ngăn xếp xóa khối `abc`)

```python
class Solution:
    def isValid(self, s: str) -> bool:
        stack = []
        for ch in s:
            stack.append(ch)
            if ch == "c" and stack[-3:] == ["a", "b", "c"]:   # ba ký tự trên cùng là "abc"
                del stack[-3:]
        return not stack
```

Công thức nhớ nhanh: hợp lệ ⇔ xóa liên tục `abc` đưa chuỗi về rỗng; dùng ngăn xếp, kiểm tra ba ký tự trên cùng mỗi khi vừa đẩy `c`.

### Checklist trước khi submit

- [ ] Đã **xóa** `abc` (chiều ngược của phép chèn), không cố mô phỏng phép chèn?
- [ ] Kiểm tra **ba ký tự liền nhau** `a, b, c` (không chỉ đếm số lượng hay tiền tố)?
- [ ] Không dùng `replace` đúng **một lần** (nếu dùng `replace`, phải lặp)?
- [ ] Trả về `not stack` (hợp lệ khi ngăn xếp rỗng)?
- [ ] Đã thử: `"abc"`, `"aabcbc"`, `"abccba"`, `"aabbcc"`, `"aaabcbcbc"`, độ dài không chia hết cho 3, chuỗi dài lồng nhau?